# RL training / validation diagnostics

This notebook is a compact reproduction of the current scheduler training and validation diagnosis. It does not launch a full training job by default. Instead, it loads an existing run, checks the training log, scheduler datasets, frozen-predictor metrics, and the feasible-subset logic used by DQN.

Default run tag: `smooth_dqn_20260421_0246`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    ROOT = Path("/home/horeb/_code/microclimate_demo/rl_sensor_scheduling_framework")
sys.path.insert(0, str(ROOT / "src"))

TAG = "smooth_dqn_20260421_0246"
SCHEDULERS = ["full_open", "dqn", "periodic", "round_robin", "info_priority", "random"]
MODELS = ["lstm", "tcn", "transformer"]

print(ROOT)
print((ROOT / "reports" / "aggregate" / f"metrics_forecast_all_{TAG}.csv").exists())

## 1. Aggregate result: DQN is the outlier

This table reproduces the high-level symptom. The failure is not a small dRMSE formatting artifact: DQN is orders of magnitude worse than rule-based schedulers in frozen-forecast evaluation.

In [ ]:
summary_path = ROOT / "reports" / "aggregate" / f"metrics_forecast_all_{TAG}_scheduler_summary.csv"
summary = pd.read_csv(summary_path)
cols = [
    "scheduler",
    "rmse_mean",
    "rmse_increase_pct_vs_full_open",
    "power_saving_pct_vs_full_open",
    "dtw_h1_increase_pct_vs_full_open",
    "pearson_h1_delta_vs_full_open",
]
summary[cols].sort_values("rmse_increase_pct_vs_full_open")

## 2. DQN policy collapse: cheap sensors stay on, snow sensors nearly disappear

The DQN dataset reveals the policy actually used on `final_test`. It mostly keeps the cheap meteorological trio and almost never uses the expensive snow sensors. This is already enough to explain why snow transport variables become fragile.

In [ ]:
def sensor_rates(tag: str, scheduler: str) -> pd.DataFrame:
    path = ROOT / "data" / "processed" / f"{tag}_{scheduler}.npz"
    data = np.load(path, allow_pickle=True)
    sensors = [str(x) for x in data["sensor_ids"]]
    out = pd.DataFrame({"sensor": sensors})
    for key in ["powered_mask", "warming_mask", "ready_mask"]:
        if key in data.files:
            out[key.replace("_mask", "_rate")] = data[key].astype(float).mean(axis=0)
    return out

pd.concat(
    [sensor_rates(TAG, sched).assign(scheduler=sched) for sched in SCHEDULERS],
    ignore_index=True,
)[["scheduler", "sensor", "powered_rate", "ready_rate"]].sort_values(["scheduler", "sensor"])

## 3. Estimator input damage: one latent channel goes far out of distribution

Here we compare scheduler-specific estimated input states against the shared truth targets on `final_test`. DQN observes `snow_particle_mean_velocity_ms` only about 6.5% of the time, and its estimator error on that channel becomes enormous. This poisoned input then reaches the frozen forecasting models.

In [ ]:
def estimator_error_table(tag: str, scheduler: str) -> pd.DataFrame:
    path = ROOT / "data" / "processed" / f"{tag}_{scheduler}.npz"
    data = np.load(path, allow_pickle=True)
    names = [str(x) for x in data["feature_names"]]
    x = data["input_series"].astype(float)
    y = data["target_series"].astype(float)
    obs = data["observed_mask"].astype(float).mean(axis=0)
    return pd.DataFrame({
        "scheduler": scheduler,
        "feature": names,
        "est_rmse": np.sqrt(np.nanmean((x - y) ** 2, axis=0)),
        "est_mae": np.nanmean(np.abs(x - y), axis=0),
        "obs_rate": obs,
    })

est_errors = pd.concat([estimator_error_table(TAG, sched) for sched in SCHEDULERS], ignore_index=True)
est_errors[est_errors.scheduler.isin(["full_open", "dqn", "periodic", "round_robin"])].sort_values(
    ["feature", "est_rmse"], ascending=[True, False]
).reset_index(drop=True)

## 4. Frozen-predictor target breakdown

The aggregate RMSE hides which target explodes. In this run, the TCN frozen predictor becomes extremely unstable under DQN inputs, especially on `wind_speed_ms`. LSTM and Transformer also degrade strongly, though less dramatically. This points to input-distribution shift, not just a single metric display issue.

In [ ]:
def per_target_forecast_rmse(tag: str, scheduler: str, model: str) -> pd.DataFrame:
    path = ROOT / "reports" / "runs" / f"{tag}_{scheduler}_pred_{model}" / "forecast_predictions.npz"
    data = np.load(path, allow_pickle=True)
    target_names = [str(x) for x in data["target_feature_names"]]
    err = data["y_pred"].astype(float) - data["y_true"].astype(float)
    rmse_all = np.sqrt(np.mean(err ** 2, axis=(0, 1)))
    rmse_h = np.sqrt(np.mean(err ** 2, axis=0))
    return pd.DataFrame({
        "scheduler": scheduler,
        "model": model,
        "target": target_names,
        "rmse_all_h": rmse_all,
        "rmse_h1": rmse_h[0],
        "rmse_h2": rmse_h[1],
        "rmse_h3": rmse_h[2],
    })

target_breakdown = pd.concat(
    [per_target_forecast_rmse(TAG, "dqn", model) for model in MODELS],
    ignore_index=True,
)
target_breakdown.sort_values("rmse_all_h", ascending=False)

## 5. Training log: oracle reward is not aligned with final frozen evaluation

DQN sees small per-step oracle `forecast_loss` during training and validation, but the final frozen-predictor evaluation is catastrophic. This is the key mismatch: the training reward oracle is not making the collapsed snow-sensor policy sufficiently expensive.

In [ ]:
train_log_path = ROOT / "reports" / "runs" / f"{TAG}_dqn" / "training_log.csv"
train_log = pd.read_csv(train_log_path)
display_cols = [
    "reward",
    "forecast_loss",
    "switch_penalty",
    "coverage_penalty",
    "power",
    "coverage",
    "val_objective",
    "val_forecast_loss",
    "val_power",
    "epsilon",
    "q_mean",
    "q_entropy",
]
train_log[[c for c in display_cols if c in train_log.columns]].tail(20)

In [ ]:
train_log[[c for c in ["forecast_loss", "switch_penalty", "coverage_penalty", "power", "coverage", "val_forecast_loss", "val_power"] if c in train_log.columns]].describe()

## 6. Bellman bootstrap check: current batched DQN target ignores previous selected subset

The online projector's feasible set depends on `prev_selected`, because startup peak power is transition-dependent. The batched DQN bootstrap currently scores candidates with `prev_selected=None`, so it does not learn the correct continuation value for actions that are only feasible, or cheaper to keep, after prior activation.

This is a code-level regression introduced by the batched target calculation. The earlier loop used the sampled action mask as previous selection when computing the next feasible set.

In [ ]:
from core.config import load_yaml
from scheduling.online_projector import OnlineSubsetProjector

sensor_cfg = load_yaml(str(ROOT / "configs" / "sensors" / "windblown_sensors.yaml"))
base_cfg = load_yaml(str(ROOT / "configs" / "base.yaml"))
sensor_ids = [str(s["sensor_id"]) for s in sensor_cfg["sensors"]]
power_costs = {str(s["sensor_id"]): float(s["power_cost"]) for s in sensor_cfg["sensors"]}
startup_costs = {str(s["sensor_id"]): float(s.get("startup_peak_power", s["power_cost"])) for s in sensor_cfg["sensors"]}
constraints = base_cfg["constraints"]
selector = OnlineSubsetProjector(
    sensor_ids=sensor_ids,
    power_costs=power_costs,
    startup_peak_costs=startup_costs,
    max_active=int(constraints["max_active"]),
    per_step_budget=float(constraints["per_step_budget"]),
    startup_peak_budget=float(constraints["startup_peak_budget"]),
    safety_margin=float(constraints.get("power_safety_margin", 0.0)),
)

cases = {
    "cold_start_prev_none": None,
    "cheap_trio_prev": ["met_station", "radiation", "surface_temp_ir"],
    "cheap_trio_plus_laser_prev": ["met_station", "radiation", "surface_temp_ir", "laser_disdrometer"],
    "cheap_trio_plus_fc4_prev": ["met_station", "radiation", "surface_temp_ir", "fc4_flux"],
}
rows = []
for name, prev in cases.items():
    for subset in selector.feasible_subsets(prev, allow_empty=False):
        rows.append({
            "case": name,
            "prev_selected": str(prev),
            "subset": "+".join(subset),
            "steady_power": selector.steady_power(subset),
            "transition_peak_power": selector.transition_peak_power(subset, prev),
            "has_laser": "laser_disdrometer" in subset,
            "has_fc4": "fc4_flux" in subset,
        })
feasible_table = pd.DataFrame(rows)
feasible_table.groupby("case").agg(
    n_feasible=("subset", "count"),
    n_with_laser=("has_laser", "sum"),
    n_with_fc4=("has_fc4", "sum"),
).reset_index()

In [ ]:
feasible_table[
    feasible_table["subset"].isin([
        "met_station+radiation+surface_temp_ir+laser_disdrometer",
        "met_station+radiation+surface_temp_ir+fc4_flux",
    ])
].sort_values(["subset", "case"])

## 7. Minimal training / validation flow

The actual pipeline is long, but the control logic is this:

1. Build `TruthReplayEnvironment`, `KalmanFilterEstimator`, `OnlineSubsetProjector`, and frozen reward oracle.
2. At each training step, flatten the estimator state and ask DQN for a feasible subset.
3. Step the environment, update Kalman state, compute frozen forecast loss, build task reward, push transition into replay.
4. Every `save_every` episodes, run deterministic validation on `rl_val` and choose checkpoint.
5. Build `final_test` dataset from the chosen checkpoint, then evaluate frozen predictors on that dataset.

The current failure happens between steps 3 and 5: the step-level oracle reward remains small for the collapsed DQN policy, but the final dataset is out-of-distribution for frozen forecasting models.

In [ ]:
def minimal_training_validation_pseudocode():
    """Reference skeleton; this cell documents the flow without launching a full experiment."""
    flow = [
        "state_vec = flatten_rl_state(estimator.get_rl_state_features())",
        "selected = agent.act(state_vec, prev_selected=prev_selected)",
        "step = env.step(selected)",
        "estimator.predict(); estimator.update(step['available_observations'])",
        "forecast_loss = frozen_oracle.score(history, future_truth, observed_mask, time_index)",
        "task_reward = -compute_forecast_task_terms(forecast_loss, penalties).task_loss",
        "agent.observe(state_vec, selected, task_reward, next_state_vec, done)",
        "if episode % save_every == 0: validate on rl_val and save best checkpoint",
        "after training: replay best checkpoint on final_test and run frozen predictor eval",
    ]
    return pd.DataFrame({"step": range(1, len(flow) + 1), "operation": flow})

minimal_training_validation_pseudocode()

## Diagnosis summary

The practical issue is not one single plot setting. The actual chain is:

- DQN converges to a low-information policy: cheap sensors almost always on, snow sensors mostly off.
- This leaves key snow/transport state channels out of distribution, especially `snow_particle_mean_velocity_ms`.
- Frozen predictor evaluation then blows up, most dramatically for TCN.
- Training-time oracle `forecast_loss` does not penalize this collapse strongly enough.
- The current batched DQN bootstrap is also technically wrong for transition-dependent feasible sets because it ignores `prev_selected`.

Immediate code fixes to test next:

- Restore transition-aware next-action bootstrapping, or store next previous-action masks in replay and batch by previous mask.
- Use `validation.objective: feasible_forecast_loss` in the actually launched base config.
- Remove non-forecast penalties from the main forecast-first run, then add warmup/switch penalties back only as controlled ablations.
- Rebalance reward oracle targets or report target-wise losses during RL training, so one collapsed target cannot hide inside a small averaged oracle score.